# 🌽 ভুট্টার রোগ নির্ণয় — MobileNetV3-Large (Kaggle → server)

**Dataset:** বাংলাদেশের দুই খেতের dataset (Seasonal Corn, MLDA) + PlantVillage/PlantDoc-এর ভুট্টা (Kaggle) — কোনোটার path খালি রাখলে সেটা বাদ — + আপনার পাতাহীন ছবি + (থাকলে) PlantVillage-এর অন্য গাছের পাতা

**এই notebook কী করে**
1. সব dataset-এর folder app-এর label-এ মেলায়: `gray_leaf_spot`, `common_rust`, `northern_leaf_blight`, `fall_armyworm`, `healthy` (শুধু একটা dataset-এ থাকা অল্প ছবির রোগ বাদ — `DROP_CLASSES`)
2. `not_leaf` class বানায়: পাতাহীন ছবি + PlantVillage-এর আলু, টমেটো ও মরিচের পাতা ("এটি ভুট্টার পাতা নয়")
3. প্রতিটা ছবি app-এর মতো করে নেয় (EXIF ঘোরানো ঠিক, বড় দিক ≤1600px, JPEG 92%) আর **একই ছবি দুবার থাকলে বাদ দেয়** — অন্য dataset-এ ছোট করে বা আবার save করে রাখা কপিও (dHash) — নইলে train আর test-এ একই ছবি গিয়ে accuracy ভুয়া বাড়ে
4. **একই পাতার augment করা কপি** (কাটা, জুম, ঘোরানো, রং বদলানো — Seasonal dataset-এ augment করা ছবি আছে) keypoint মিলিয়ে খুঁজে এক দলে রাখে, প্রতি দলে সর্বোচ্চ ৩টা
5. ~70/15/15 ভাগ (রোগ + dataset মিলিয়ে stratified; একই পাতার দল কখনো দুই ভাগে যায় না)
6. MobileNetV3-Large train: প্রতি epoch-এ নতুন করে augmentation (ঘোরানো, জুম, আলো, **ঝাপসা, noise**) + MixUp + class weight; পুরো network fine-tune
7. Test-এ মূল্যায়ন — **প্রতিটা dataset আলাদা, আর not_leaf বাদ দিয়ে শুধু ভুট্টা পাতার accuracy**
8. **`maize_server_bundle.zip`** বানায় → খুললেই `maize/model.tflite` (~৬ MB), `labels.txt`, `meta.json`

> Kaggle Notebook → Settings → **Accelerator: GPU (T4/P100)** আর **Internet: On** করে নিন।
>
> **Add Input**-এ PlantVillage (`emmarex/plantdisease`) যোগ করলে অন্য গাছের পাতাও `not_leaf`-এ যাবে; না থাকলে notebook সেটা বাদ দিয়ে চলবে।
>
> লম্বা training-এ **Save Version → Save & Run All (Commit)** দিন — বিদ্যুৎ/internet চলে গেলেও Kaggle-এর server-এ চলতে থাকবে।
>
> মাঝপথে বন্ধ হয়ে গেলে: আগের version-এর output (**Add Input → Your Work**) যোগ করে `RESUME = True` — `maize_best_mobilenetv3large.keras` নিজেই খুঁজে নিয়ে training ছাড়াই মূল্যায়ন ও export।

## ১. Setup ও Config

In [ ]:
import os, io, gc, random, json, shutil, zipfile, hashlib
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ---------------- আপনার dataset path ----------------
# উৎসের নাম → folder (ভেতরে প্রতি রোগের একটা folder)
DATASETS = {
    # নাটোর, বাংলাদেশ (vy629dngm8) — Gray_leaf_spot, Healthy, Common_rust, Bacterial Leaf Streak,
    # Maize Chlorotic Mottle Virus; augment করা কপি থাকতে পারে, নিচে এক দলে রাখা হয়
    "seasonal": "/kaggle/input/datasets/jihadulislam1/seasonal-corn-leaf-disease-dataset-a-multi-year/Final corn dataset",
    # ঝিনাইদহ, বাংলাদেশ (MLD 3myfctrgk3) — Fall_Armyworm, Gray_Spot, Healthy, Leaf_Blight
    "mlda": "/kaggle/input/datasets/jihadulislam1/mlda-multi-class-maize-leaf-disease-image-dataset/Maize dataset",
    # PlantVillage + PlantDoc — Blight, Common_Rust, Gray_Leaf_Spot, Healthy (ল্যাবের ছবি বেশি)
    "plantvillage": "/kaggle/input/datasets/smaranjitghose/corn-or-maize-leaf-disease-dataset/data",
}

# "পাতা নয়" ছবি (মাটি, হাত, আকাশ, অন্য জিনিস) — ভেতরের সব ছবি নেবে
NOT_LEAF_DIR = "/kaggle/input/datasets/jihadulislam1/without-leaves-full"
NOT_LEAF_MAX = 500           # এর বেশি থাকলে এলোমেলো ভাবে এতগুলো; বেশি দিলে রোগগুলোকে ছাপিয়ে যায়

# PlantVillage input-এ থাকলে আলু, টমেটো ও মরিচের পাতাও not_leaf-এ যোগ হবে, যাতে অন্য গাছের
# পাতা দিলে model "ভুট্টার পাতা নয়" বলে। প্রতি folder থেকে সর্বোচ্চ এতগুলো (~১৫টা folder)।
PV_BASE = "/kaggle/input/datasets/emmarex/plantdisease/PlantVillage"
OTHER_CROPS_AS_NOT_LEAF = True
OTHER_CROP_PER_FOLDER = 15   # ~২২৫টা; বেশি দিলে not_leaf রোগগুলোকে ছাপিয়ে যায়

# যে class বাদ দিতে চান (app-এর label নামে)। এই দুটো শুধু Seasonal dataset-এ, অল্প ছবি
# (~১৯০ আর ~৯০) — একটা খেতের ছবি থেকে শিখলে রোগ না চিনে খেতের পটভূমি চেনে। রাখতে চাইলে [] দিন
DROP_CLASSES = ["bacterial_leaf_streak", "maize_chlorotic_mottle_virus"]

# app ছবি বড় দিক ≤1600px আর JPEG 92% করে পাঠায়; training ছবিও তাই করে নেওয়া হয়
MAX_SIDE = 1600

# দুটো পাতার ছবির ছাপ (dHash, ৬৪ bit) এর চেয়ে কম আলাদা হলে একই ছবি ধরা হয় (ছোট করা,
# ঘোরানো বা উল্টানো কপি)। ৩-এ ভুল মিল প্রায় হয় না; 0 দিলে শুধু হুবহু এক ছবি
NEAR_DUP_BITS = 3

# augment করা কপি (কাটা, জুম, একটু ঘোরানো, রং বদলানো) dHash ধরে না; keypoint (ORB) মিলিয়ে
# অন্তত এতগুলো বিন্দু মিললে একই পাতা। পরীক্ষায় ৬০-এ ~৯৭% কপি ধরা পড়েছে, আলাদা পাতা একটাও না
ORB_MIN_INLIERS = 60
# একই পাতার সর্বোচ্চ এতগুলো কপি রাখা হয় — বাকিগুলো নতুন কিছু শেখায় না, শুধু class-কে ফোলায়
MAX_PER_GROUP = 3

IMG = 256
BATCH = 32
EPOCHS_HEAD = 8             # শুধু নতুন layer train
EPOCHS_FT = 40               # fine-tuning (early stopping আছে)
# EfficientNet-ও তুলনা করতে চাইলে নিচের লাইনের comment তুলে দিন (SPECS-এও)।
# CANDIDATES = ["efficientnetv2b2", "mobilenetv3large"]
CANDIDATES = ["mobilenetv3large"]
LABEL_SMOOTHING = 0.1        # overfitting কমায়
MIXUP_ALPHA = 0.2            # 0 দিলে MixUp বন্ধ

# ---- আগের save করা model থেকে চালু (training বাদ) ----
# আগের run-এর output (maize_best_mobilenetv3large.keras আর outputs/split_*.csv) input-এ
# যোগ করুন: Add Input → Your Work → এই notebook। True হলে training বাদ, সরাসরি মূল্যায়ন ও export;
# checkpoint না পেলে নিজে থেকেই নতুন করে train হবে।
RESUME = False
# checkpoint-এর folder; খালি রাখলে পুরো /kaggle/input-এ খুঁজবে
RESUME_DIR = ""

RESUME_MODEL = ""
if RESUME:
    search = RESUME_DIR if os.path.isdir(RESUME_DIR) else "/kaggle/input"
    # শুধু CANDIDATES-এর model-এর checkpoint (maize_best_mobilenetv3large.keras), ভুট্টারটা আগে
    hits = sorted((p for p in Path(search).rglob("*.keras")
                   if any(c in p.name.lower() for c in CANDIDATES)),
                  key=lambda p: ("maize" not in p.name.lower(), str(p)))
    if hits:
        RESUME_MODEL = str(hits[0])
        RESUME_DIR = str(hits[0].parent.parent if hits[0].parent.name == "outputs" else hits[0].parent)
        print("⏩ আগের model থেকে চালু:", RESUME_MODEL)
    else:
        print(f"⚠️ {CANDIDATES}-এর .keras পাওয়া যায়নি — নতুন করে train হবে")
        RESUME = False

WORK = Path("/kaggle/working")
OUT = WORK / "outputs"          # সব ফলাফল এখানে জমা হয়, শেষে এক zip-এ
OUT.mkdir(exist_ok=True)
CACHE = WORK / "cache"          # app-এর মতো করে নেওয়া ছবি
CACHE.mkdir(exist_ok=True)

def savefig(name):
    plt.savefig(OUT / name, dpi=150, bbox_inches="tight")
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## ২. Dataset খুঁজে বের করা
Folder-এর নামের মূল শব্দ দেখে label: `Gray_Spot` আর `Gray_Leaf_Spot` দুটোই → `gray_leaf_spot`; `Blight` আর `Leaf_Blight` → `northern_leaf_blight`।

In [ ]:
# app: lib/data/library/maize_disease_info.dart, server: models/maize/labels.txt
# ক্রম জরুরি: আগের নিয়ম আগে মেলে ("Healthy Leaf"-এ "leaf" আছে, তাই healthy আগে)
# Blight = Northern Leaf Blight (লম্বা চুরুটের মতো ধূসর-বাদামি দাগ) — নিচের ছবিতে মিলিয়ে নিন
RULES = [("healthy", "healthy"), ("armyworm", "fall_armyworm"), ("rust", "common_rust"),
         ("gray", "gray_leaf_spot"), ("grey", "gray_leaf_spot"), ("blight", "northern_leaf_blight"),
         ("streak", "bacterial_leaf_streak"), ("mottle", "maize_chlorotic_mottle_virus")]

def label_for(folder_name):
    n = folder_name.lower()
    return next((label for key, label in RULES if key in n), None)

def list_images(folder):
    return [p for p in Path(folder).rglob("*") if p.suffix.lower() in EXTS]

def class_dirs(folder):
    return [c for c in sorted(Path(folder).iterdir()) if c.is_dir() and label_for(c.name)]

def find_dataset(path, hint):
    # path-এর ভেতরে (না পেলে /kaggle/input-এ) প্রথম যে folder-এ ≥৪টা রোগের folder আছে;
    # Augmented আর PlantVillage (Tomato_Leaf_Mold ইত্যাদি) বাদ
    base = path if path and os.path.isdir(path) else "/kaggle/input"
    if base != path:
        print(f"⚠️ {path} পাওয়া যায়নি, /kaggle/input-এ '{hint}' খুঁজছি...")
    for root, dirs, _ in os.walk(base):
        low = root.lower()
        if "augment" in low or (base != path and hint not in low):
            continue
        if sum(label_for(d) is not None for d in dirs) >= 4 and \
                not any(d.lower().startswith(("tomato", "potato", "pepper")) for d in dirs):
            return root
    raise AssertionError(f"dataset পাওয়া যায়নি — DATASETS-এ path ঠিক করুন ({hint})")

HINTS = {"seasonal": "seasonal", "mlda": "mlda", "plantvillage": "corn-or-maize"}
for src in list(DATASETS):
    if not DATASETS[src] and src not in HINTS:
        print(f"\n{src}: path খালি — বাদ")
        del DATASETS[src]
        continue
    DATASETS[src] = find_dataset(DATASETS[src], HINTS.get(src, src))
    print(f"\n{src}: {DATASETS[src]}")
    for c in sorted(Path(DATASETS[src]).iterdir()):
        if c.is_dir():
            print(f"  {c.name:35s} -> {label_for(c.name) or '— (বাদ)'}")

assert os.path.isdir(NOT_LEAF_DIR), f"NOT_LEAF_DIR পাওয়া যায়নি: {NOT_LEAF_DIR}"

PV_ROOT = None
if OTHER_CROPS_AS_NOT_LEAF:
    search_base = PV_BASE if os.path.exists(PV_BASE) else "/kaggle/input"
    for root, dirs, _ in os.walk(search_base):
        if sum(d.lower().startswith("tomato") for d in dirs) >= 5:
            PV_ROOT = root
            break
    print("\nPlantVillage:", PV_ROOT or "নেই — অন্য গাছের পাতা not_leaf-এ যাবে না")

## ৩. ছবি সংগ্রহ, app-এর মতো করে নেওয়া, duplicate বাদ

In [ ]:
records = []   # (মূল path, label, source)
for src, root in DATASETS.items():
    for c in class_dirs(root):
        label = label_for(c.name)
        if label not in DROP_CLASSES:
            records += [(str(p), label, src) for p in list_images(c)]

rng = random.Random(SEED)
if PV_ROOT:
    for d in sorted(Path(PV_ROOT).iterdir()):
        if d.is_dir() and d.name.lower().startswith(("potato", "tomato", "pepper")):
            imgs = list_images(d)
            rng.shuffle(imgs)
            records += [(str(p), "not_leaf", "other_crop") for p in imgs[:OTHER_CROP_PER_FOLDER]]

no_leaf = list_images(NOT_LEAF_DIR)
rng.shuffle(no_leaf)
records += [(str(p), "not_leaf", "no_leaf") for p in no_leaf[:NOT_LEAF_MAX]]
df = pd.DataFrame(records, columns=["orig_path", "label", "source"])
print("মোট ছবি:", len(df))

FLIPS = [None, Image.FLIP_LEFT_RIGHT, Image.FLIP_TOP_BOTTOM, Image.ROTATE_90,
         Image.ROTATE_180, Image.ROTATE_270, Image.TRANSPOSE, Image.TRANSVERSE]

def dhash(im):
    # ছবির "আঙুলের ছাপ": ছোট করা বা আবার save করা কপিরও প্রায় একই থাকে।
    # ৮ রকম ঘোরানো/উল্টানো রূপের ছাপ, কমা দিয়ে — augment করা কপিও ধরা পড়ে
    g0 = im.convert("L")
    out = []
    for f in FLIPS:
        g = np.asarray((g0 if f is None else g0.transpose(f)).resize((9, 8), Image.BILINEAR), dtype=np.int16)
        out.append("".join("1" if b else "0" for b in (g[:, 1:] > g[:, :-1]).ravel()))
    return ",".join(out)

def prepare(path):
    # server-এর মতো EXIF অনুযায়ী ঘোরানো, RGB; app-এর মতো বড় দিক ≤ MAX_SIDE, JPEG 92%
    raw = Path(path).read_bytes()
    dst = CACHE / f"{hashlib.md5(raw).hexdigest()}.jpg"
    try:
        if not dst.exists():
            im = ImageOps.exif_transpose(Image.open(io.BytesIO(raw))).convert("RGB")
            im.thumbnail((MAX_SIDE, MAX_SIDE), Image.LANCZOS)
            # আগে আলাদা নামে লিখে তারপর নাম বদল, যাতে একই ছবির আরেক thread অর্ধেক-লেখা ফাইল না পড়ে
            part = dst.with_name(f"{dst.stem}.{id(im)}.part.jpg")
            im.save(part, quality=92)
            os.replace(part, dst)
        # সবসময় save করা ফাইল থেকে, যাতে একই ছবির dHash হুবহু এক হয়
        return dhash(Image.open(dst)), str(dst)
    except Exception as e:
        print("পড়া যায়নি, বাদ:", path, e)
        return None, None

with ThreadPoolExecutor(8) as pool:
    df["dhash"], df["path"] = zip(*pool.map(prepare, df["orig_path"]))

bad = df["path"].isna()
N_BAD = int(bad.sum())
df = df[~bad]
df = df.reset_index(drop=True)

# পাতার ছবিতে কোনো ঘোরানো/উল্টানো রূপের dHash ≤ NEAR_DUP_BITS আলাদা হলে একই ছবি।
# not_leaf-এ শুধু হুবহু মিল — আকাশ/দেয়ালের ছবি এমনিতেই কাছাকাছি দেখায়।
idx = np.flatnonzero((df["label"] != "not_leaf").to_numpy())
bits = np.array([[[c == "1" for c in h] for h in df.at[i, "dhash"].split(",")] for i in idx],
                dtype=np.float32).reshape(-1, 8, 64)
base, every = bits[:, 0], bits.reshape(-1, 64)
near = np.zeros((len(idx), len(idx)), dtype=bool)
for s in range(0, len(idx), 500):
    blk = base[s:s + 500]
    dist = 64 - (blk @ every.T + (1 - blk) @ (1 - every).T)
    near[s:s + 500] = dist.reshape(len(blk), len(idx), 8).min(-1) <= NEAR_DUP_BITS
near |= near.T
# প্রতিটা ছবি আগের প্রথম "মূল" ছবির দলে; শিকলের মতো জুড়ে বড় দল হয় না
keeper = np.arange(len(idx))
for k in range(len(idx)):
    hit = np.flatnonzero(near[k, :k] & (keeper[:k] == np.arange(k)))
    if len(hit):
        keeper[k] = hit[0]
df["group"] = df["dhash"].map(lambda h: "nl_" + h)
df.loc[idx, "group"] = [str(idx[j]) for j in keeper]

# একই ছবি দুই ভিন্ন রোগে থাকলে কোনটা ঠিক জানা নেই → সব কপি বাদ; একই রোগে থাকলে একটা রাখা
conflict = df.groupby("group")["label"].transform("nunique") > 1
dup = df.duplicated("group") & ~conflict
cross = df[df.duplicated("group", keep=False)].groupby("group")["source"].nunique().gt(1).sum()
print(f"নষ্ট: {N_BAD} | একই ছবি একাধিকবার: {int(dup.sum())} (দুই dataset-এ একই: {int(cross)}) "
      f"| ভিন্ন রোগে একই ছবি: {int(conflict.sum())}")
df = df[~conflict & ~dup].reset_index(drop=True)
N_DUP = int(dup.sum() + conflict.sum())

### একই পাতার augment করা কপি এক দলে
Seasonal dataset-এ একই পাতার কাটা/জুম করা/ঘোরানো/রং বদলানো কপি থাকতে পারে, আর PlantVillage-এর ছবি অন্য dataset-এও থাকতে পারে। এক কপি train-এ আর আরেকটা test-এ গেলে model মুখস্থ করেই পাস করে — accuracy ভুয়া বাড়ে, মাঠে গিয়ে কমে। তাই keypoint মিলিয়ে একই পাতার ছবি এক দলে রাখা হয়, **পুরো দল একসাথে train বা val বা test-এ যায়**, আর প্রতি দল থেকে সর্বোচ্চ `MAX_PER_GROUP`টা কপি থাকে।

In [ ]:
leaf_i = np.flatnonzero((df["label"] != "not_leaf").to_numpy())
SHORTLIST = min(15, len(leaf_i) - 1)

# ধাপ ১: ImageNet-এর ছাপ (৮ রকম ঘোরানো/উল্টানো রূপ) দিয়ে প্রতিটা ছবির কাছের SHORTLISTটা ছবি
emb_net = keras.applications.MobileNetV3Large(input_shape=(224, 224, 3), include_top=False,
                                              pooling="avg", weights="imagenet", include_preprocessing=True)
def load224(p):
    return tf.image.resize(tf.io.decode_jpeg(tf.io.read_file(p), channels=3), (224, 224))

def dihedral(x):
    t = tf.transpose(x, [0, 2, 1, 3])
    return [x, tf.reverse(x, [2]), tf.reverse(x, [1]), tf.image.rot90(x, 1), tf.image.rot90(x, 2),
            tf.image.rot90(x, 3), t, tf.image.rot90(t, 2)]

emb = []
for x in tf.data.Dataset.from_tensor_slices(df.loc[leaf_i, "path"].values).map(load224).batch(256):
    emb.append(np.stack([emb_net.predict(v, batch_size=32, verbose=0) for v in dihedral(x)], 1))
emb = np.concatenate(emb)
emb /= np.linalg.norm(emb, axis=-1, keepdims=True)
del emb_net; gc.collect()

base_e, every_e = emb[:, 0], emb.reshape(-1, emb.shape[-1])
pairs = set()
for s in range(0, len(leaf_i), 500):
    sim = (base_e[s:s + 500] @ every_e.T).reshape(-1, len(leaf_i), 8).max(-1)
    for r, row in enumerate(sim):
        row[s + r] = -1
        pairs |= {tuple(sorted((s + r, int(j)))) for j in np.argpartition(-row, SHORTLIST)[:SHORTLIST]}
pairs = sorted(pairs)

# ধাপ ২: ORB keypoint মিলিয়ে নিশ্চিত — জুম/ঘোরানো/রং বদলালেও একই ছবির বিন্দুগুলো মেলে।
# উল্টানো কপির জন্য আয়নার ছবিরও বিন্দু রাখা হয়
def orb_feats(path):
    g = np.asarray(Image.open(path).convert("L"))
    s = 512 / max(g.shape)
    g = cv2.resize(g, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    out = []
    for gg in (g, np.ascontiguousarray(g[:, ::-1])):
        k, d = cv2.ORB_create(1000).detectAndCompute(gg, None)
        out.append((np.float32([p.pt for p in k]) if k else None, d))
    return out

def orb_inliers(a, b):
    best, bf = 0, cv2.BFMatcher(cv2.NORM_HAMMING)
    ka, da = FEATS[a][0]
    for kb, db in FEATS[b]:
        if da is None or db is None or len(da) < 8 or len(db) < 8:
            continue
        m = [x[0] for x in bf.knnMatch(da, db, k=2) if len(x) == 2 and x[0].distance < 0.8 * x[1].distance]
        if len(m) < 6:
            continue
        _, inl = cv2.estimateAffinePartial2D(ka[[x.queryIdx for x in m]], kb[[x.trainIdx for x in m]],
                                             method=cv2.RANSAC, ransacReprojThreshold=5)
        if inl is not None:
            best = max(best, int(inl.sum()))
    return best

# একটার পর একটা (~১০ মিনিট) — OpenCV-কে thread-এ চালালে মাঝে মাঝে crash করে
FEATS = [orb_feats(p) for p in df.loc[leaf_i, "path"]]
score = [orb_inliers(a, b) for a, b in pairs]

same = [(a, b, n) for (a, b), n in zip(pairs, score) if n >= ORB_MIN_INLIERS]
parent = list(range(len(leaf_i)))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]
        a = parent[a]
    return a
for a, b, _ in same:
    parent[find(a)] = find(b)
df.loc[leaf_i, "group"] = ["leaf_" + str(find(k)) for k in range(len(leaf_i))]

sizes = df.loc[leaf_i].groupby("group").size()
mixed = int(df.loc[leaf_i].groupby("group")["label"].nunique().gt(1).sum())
N_SAME = len(same)
print(f"একই পাতার কপি-জোড়া: {N_SAME} | কপিওয়ালা দল: {int((sizes > 1).sum())}, সবচেয়ে বড় দল: {sizes.max()}টা ছবি")
print("কোন dataset-এর কত ছবির কপি আছে:", df.loc[leaf_i][df.loc[leaf_i, "group"].map(sizes) > 1]
      .source.value_counts().to_dict())
if mixed:
    print(f"⚠️ {mixed}টা দলে ভিন্ন রোগের label — একই পাতা দুই রোগে; নিচের ছবিতে দেখে নিন")

# সীমানার কাছের মিলগুলো (সবচেয়ে কম বিন্দু) দেখুন — সত্যিই একই পাতা কি না
show = sorted(same, key=lambda t: t[2])[:8]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 2.6 * len(show)), squeeze=False)
    for r, (a, b, n) in enumerate(show):
        for k, i in enumerate((a, b)):
            row = df.loc[leaf_i[i]]
            axes[r, k].imshow(Image.open(row.path).convert("RGB")); axes[r, k].axis("off")
            axes[r, k].set_title(f"{row.label} | {row.source}" + (f" | {n} বিন্দু" if k == 0 else ""), fontsize=8)
    plt.tight_layout(); savefig("01b_same_leaf_pairs.png"); plt.show()

before = df.label.value_counts()
df = (df.sample(frac=1, random_state=SEED).groupby("group", sort=False).head(MAX_PER_GROUP)
      .sort_index().reset_index(drop=True))
print(f"\nপ্রতি দলে সর্বোচ্চ {MAX_PER_GROUP}টা কপি রেখে বাদ: {int(before.sum() - len(df))}টা")
print(pd.DataFrame({"আগে": before, "এখন": df.label.value_counts(),
                    "আলাদা পাতা": df.groupby("label")["group"].nunique()}))

CLASSES = sorted(df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
df["y"] = df["label"].map(CLASS_TO_IDX)
SOURCE_ORDER = [s for s in [*DATASETS, "other_crop", "no_leaf"] if s in set(df.source)]
print("\nClasses:", CLASSES)
print(pd.crosstab(df["label"], df["source"], margins=True))
few = df[df.label != "not_leaf"].label.value_counts()
few = few[few < 60]
if len(few):
    print(f"\n⚠️ খুব কম ছবি: {few.to_dict()} — model এগুলো ভালো শিখবে না; "
          "নতুন dataset যোগ করুন, নয়তো DROP_CLASSES-এ দিন")

In [ ]:
# প্রতি রোগের প্রতিটা dataset থেকে ২টা ছবি — একই রোগ সব dataset-এ একই রকম দেখায় কি না যাচাই
# (বিশেষ করে mlda-র Leaf_Blight আর plantvillage-এর Blight: দুটোই লম্বা চুরুটের মতো দাগ হওয়া উচিত)
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(14, 2.4 * len(CLASSES)), squeeze=False)
for r, c in enumerate(CLASSES):
    part = df[df.label == c]
    sample = part.groupby("source", group_keys=False).apply(
        lambda g: g.sample(min(2, len(g)), random_state=1)).head(6)
    for k in range(6):
        axes[r, k].axis("off")
    for k, (p, s) in enumerate(zip(sample["path"], sample["source"])):
        axes[r, k].imshow(Image.open(p).convert("RGB"))
        axes[r, k].set_title(f"{c} | {s}", fontsize=7, loc="left")
plt.tight_layout(); savefig("01_class_samples.png"); plt.show()

## ৪. Train / Val / Test ভাগ
~70/15/15, রোগ আর dataset মিলিয়ে stratified — প্রতিটা ভাগে প্রতিটা dataset-এর প্রতিটা রোগ থাকে। **একই পাতার দল পুরোটা এক ভাগে**, তাই test-এর পাতা training-এ কোনো রূপে দেখা হয়নি। Augmentation শুধু train-এ।

In [ ]:
# RESUME-এ আগের run-এর split_*.csv পাওয়া গেলে হুবহু সেই ভাগ, নইলে একই seed দিয়ে আবার ভাগ
def prev_split(n):
    return next(Path(RESUME_DIR).rglob(f"split_{n}.csv"), None) if RESUME else None

prev = {n: prev_split(n) for n in ("train", "val", "test")}
if all(prev.values()):
    train_df, val_df, test_df = (pd.read_csv(prev[n]) for n in ("train", "val", "test"))
    for _p in (train_df, val_df, test_df):
        _p["y"] = _p["label"].map(CLASS_TO_IDX)
    assert all(os.path.exists(p) for p in test_df["path"][:20]), "split_*.csv-এর path এই session-এ নেই"
    print("আগের run-এর split_*.csv থেকে একই ভাগ নেওয়া হলো")
else:
    strata = (df["label"] + "|" + df["source"]).values
    groups = df["group"].values
    # আগে ১/৭ (~১৪%) test, বাকিটার ১/৬ (~১৪%) val — দল ভাঙে না
    rest_i, test_i = next(StratifiedGroupKFold(7, shuffle=True, random_state=SEED).split(df, strata, groups))
    tr_i, va_i = next(StratifiedGroupKFold(6, shuffle=True, random_state=SEED)
                      .split(rest_i, strata[rest_i], groups[rest_i]))
    train_df, val_df, test_df = (df.iloc[i].reset_index(drop=True)
                                 for i in (rest_i[tr_i], rest_i[va_i], test_i))
    assert not (set(train_df.group) & set(val_df.group) or set(train_df.group) & set(test_df.group)
                or set(val_df.group) & set(test_df.group)), "একই পাতা দুই ভাগে"

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, len(part))
counts = pd.DataFrame({
    "train": train_df.label.value_counts(),
    "val": val_df.label.value_counts(),
    "test": test_df.label.value_counts(),
})
print(counts)
print("\nTest-এ কোন উৎসের কত ছবি:")
print(pd.crosstab(test_df["label"], test_df["source"]))

counts.to_csv(OUT / "class_counts.csv")
for _n, _p in [("train", train_df), ("val", val_df), ("test", test_df)]:
    _p.to_csv(OUT / f"split_{_n}.csv", index=False)

## ৫. tf.data pipeline + augmentation + MixUp
Label এখন one-hot (label smoothing আর MixUp-এর জন্য দরকার)।

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
N_CLASSES = len(CLASSES)

def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG, IMG))          # 0..255 float
    return tf.cast(img, tf.float32), label

# মাঠের অবস্থা অনুকরণ: ঘোরানো, জুম, আলো-ছায়া, কনট্রাস্ট
augment = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom((-0.15, 0.1)),   # বেশি জুমে দাগ কেটে যায়
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.3, value_range=(0, 255)),
    layers.RandomContrast(0.3),
], name="augment")

def field_noise(x):
    # ৩০% ছবি ঝাপসা (অর্ধেক করে আবার বড়), প্রতিটায় ০–১২ মাত্রার noise
    n = tf.shape(x)[0]
    soft = tf.image.resize(tf.image.resize(x, (IMG // 2, IMG // 2)), (IMG, IMG))
    blur = tf.cast(tf.random.uniform([n, 1, 1, 1]) < 0.3, x.dtype)
    x = blur * soft + (1 - blur) * x
    x = x + tf.random.normal(tf.shape(x)) * tf.random.uniform([n, 1, 1, 1], 0., 12.)
    return tf.clip_by_value(x, 0., 255.)

def sample_beta(alpha, n):
    g1 = tf.random.gamma([n], alpha)
    g2 = tf.random.gamma([n], alpha)
    return g1 / (g1 + g2)

# MixUp: দুটো ছবি হালকা মিশিয়ে label-ও মেশায়, model অতি-আত্মবিশ্বাসী হয় না
def mixup(x, y):
    n = tf.shape(x)[0]
    lam = sample_beta(MIXUP_ALPHA, n)
    idx = tf.random.shuffle(tf.range(n))
    lx = tf.reshape(lam, [-1, 1, 1, 1])
    ly = tf.reshape(lam, [-1, 1])
    return x * lx + tf.gather(x, idx) * (1 - lx), y * ly + tf.gather(y, idx) * (1 - ly)

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices(
        (frame["path"].values, frame["y"].values.astype("int32")))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (x, tf.one_hot(y, N_CLASSES)), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH)
    if training:
        ds = ds.map(lambda x, y: (field_noise(tf.clip_by_value(augment(x, training=True), 0., 255.)), y),
                    num_parallel_calls=AUTOTUNE)
        if MIXUP_ALPHA > 0:
            ds = ds.map(mixup, num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_df, True)
val_ds = make_ds(val_df, False)
test_ds = make_ds(test_df, False)

# Class imbalance ঠিক করতে weight (one-hot label-এও কাজ করে)
cw = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=train_df["y"].values)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print("Class weights:", {CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

# Augmentation + MixUp কেমন দেখায়
x, y = next(iter(train_ds))
plt.figure(figsize=(10, 3))
for i in range(5):
    plt.subplot(1, 5, i + 1)
    top = int(tf.argmax(y[i]))
    plt.imshow(x[i].numpy().astype("uint8"))
    plt.title(f"{CLASSES[top]} ({float(y[i][top]):.0%})", fontsize=7); plt.axis("off")
savefig("02_augmentation_preview.png"); plt.show()

## ৬. Model
`include_preprocessing=True`, তাই **ইনপুট 0–255 float** (app-এ `/255` করা যাবে না)।

In [ ]:
SPECS = {
    # "efficientnetv2b2": dict(ctor=keras.applications.EfficientNetV2B2, unfreeze=200, lr_ft=5e-5),
    "mobilenetv3large": dict(ctor=keras.applications.MobileNetV3Large, unfreeze=999, lr_ft=5e-5),
}

def build_model(name):
    base = SPECS[name]["ctor"](
        input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet",
        include_preprocessing=True)
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3), name="image")   # 0..255 float32
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(N_CLASSES, activation="softmax", name="probs")(x)
    return keras.Model(inp, out, name=f"maize_{name}"), base

def compile_model(m, lr):
    m.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=[keras.metrics.CategoricalAccuracy(name="accuracy")],
    )

def train_candidate(name):
    spec = SPECS[name]
    m, base = build_model(name)

    # ধাপ ১: শুধু নতুন head
    compile_model(m, 1e-3)
    h1 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=2, verbose=1),
               ])

    # ধাপ ২: fine-tuning, BatchNorm frozen
    base.trainable = True
    for layer in base.layers[:-spec["unfreeze"]]:
        layer.trainable = False
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(m, spec["lr_ft"])
    ckpt = str(WORK / f"maize_best_{name}.keras")
    h2 = m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
               class_weight=CLASS_WEIGHT, callbacks=[
                   keras.callbacks.ModelCheckpoint(ckpt, monitor="val_accuracy",
                                                   save_best_only=True, verbose=1),
                   keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=8,
                                                 restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                                     patience=3, verbose=1),
               ])
    return m, h1, h2, ckpt

# TTA: আসল + বাম-ডান উল্টানো + উপর-নিচ উল্টানো, তিনটার গড়
def predict_tta(m, frame):
    ds = make_ds(frame, False)
    p = m.predict(ds, verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0)
    p += m.predict(ds.map(lambda x, y: (tf.image.flip_up_down(x), y)), verbose=0)
    return p / 3

## ৭. Training
GPU-তে MobileNet ~২০–৩০ মিনিট। `RESUME`-এ আগের checkpoint পেলে training বাদ।

In [ ]:
def load_saved(path):
    # আগের run-এর ModelCheckpoint (.keras); না খুললে একই model বানিয়ে শুধু weights বসানো
    stem = Path(path).stem.lower()
    try:
        m = keras.models.load_model(path, compile=False)
    except Exception as e:
        print("load_model হয়নি, weights আলাদা করে বসাচ্ছি:", e)
        m, _ = build_model(next(k for k in SPECS if k in stem))
        m.load_weights(path)
    name = next((k for k in SPECS if k in f"{stem} {m.name}".lower()), stem)
    assert m.output_shape[-1] == N_CLASSES, \
        f"model-এ {m.output_shape[-1]}টা class, data-তে {N_CLASSES}টা — dataset/DROP_CLASSES আগের মতো রাখুন"
    return name, m

results = {}
y_val = val_df["y"].values

def evaluate_candidate(name, m, h1_, h2_, ckpt):
    val_plain = float((m.predict(val_ds, verbose=0).argmax(1) == y_val).mean())
    val_tta = float((predict_tta(m, val_df).argmax(1) == y_val).mean())
    results[name] = dict(model=m, h1=h1_, h2=h2_, ckpt=ckpt,
                         val_plain=val_plain, val_tta=val_tta)
    print(f"{name}: val {val_plain*100:.2f}% | val+TTA {val_tta*100:.2f}%")

if RESUME:
    name, m = load_saved(RESUME_MODEL)
    print("⏩ training বাদ — আগের model:", name)
    evaluate_candidate(name, m, None, None, RESUME_MODEL)
else:
    for name in CANDIDATES:
        print("\n" + "=" * 25, name, "=" * 25)
        evaluate_candidate(name, *train_candidate(name))

## ৮. ফলাফল (একাধিক model থাকলে তুলনা ও বাছাই)
বাছাই **val** দিয়ে। Test শুধু দেখানোর জন্য, তাই test সংখ্যা সৎ থাকে।

In [ ]:
y_test = test_df["y"].values
rows = []
for name, r in results.items():
    t_plain = float((r["model"].predict(test_ds, verbose=0).argmax(1) == y_test).mean())
    t_tta = float((predict_tta(r["model"], test_df).argmax(1) == y_test).mean())
    r["test_plain"], r["test_tta"] = t_plain, t_tta
    rows.append({"model": name, "params_M": round(r["model"].count_params() / 1e6, 2),
                 "val %": round(r["val_plain"] * 100, 2), "val+TTA %": round(r["val_tta"] * 100, 2),
                 "test %": round(t_plain * 100, 2), "test+TTA %": round(t_tta * 100, 2)})
comparison = pd.DataFrame(rows)
comparison.to_csv(OUT / "model_comparison.csv", index=False)
print(comparison.to_string(index=False))

BEST_NAME = max(results, key=lambda n: results[n]["val_plain"])
best = results[BEST_NAME]
model, h1, h2, BEST = best["model"], best["h1"], best["h2"], best["ckpt"]
# server প্রতি ছবি একবারই চালায়, তাই TTA শুধু তুলনার জন্য মাপা, export-এ বন্ধ
USE_TTA_FINAL = False
print(f"\nসেরা model: {BEST_NAME} | TTA: {'চালু' if USE_TTA_FINAL else 'বন্ধ'}")

In [ ]:
if h1 is None:
    print("আগের save করা model — training history নেই, তাই chart বাদ")
else:
    acc = h1.history["accuracy"] + h2.history["accuracy"]
    val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
    loss = h1.history["loss"] + h2.history["loss"]
    val_loss = h1.history["val_loss"] + h2.history["val_loss"]
    split = len(h1.history["accuracy"])

    pd.DataFrame({"accuracy": acc, "val_accuracy": val_acc, "loss": loss, "val_loss": val_loss}).to_csv(
        OUT / "training_history.csv", index_label="epoch")

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="val")
    ax[0].axvline(split - 1, ls="--", c="gray"); ax[0].set_title(f"Accuracy ({BEST_NAME})"); ax[0].legend()
    ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="val")
    ax[1].axvline(split - 1, ls="--", c="gray"); ax[1].set_title("Loss"); ax[1].legend()
    savefig("03_training_curves.png"); plt.show()

## ৯. Test set-এ সৎ মূল্যায়ন
Test ছবি training-এ কখনো দেখেনি। **এই সংখ্যাটাই আসল।**

In [ ]:
y_true = test_df["y"].values
probs_plain = model.predict(test_ds, verbose=0)
probs = predict_tta(model, test_df) if USE_TTA_FINAL else probs_plain
print("Model:", BEST_NAME, "| TTA:", USE_TTA_FINAL)
y_pred = probs.argmax(1)
conf = probs.max(1)

print("Test accuracy:", round((y_pred == y_true).mean() * 100, 2), "%\n")
report_txt = classification_report(y_true, y_pred, target_names=CLASSES, digits=3)
print(report_txt)
TEST_ACC = float((y_pred == y_true).mean())

# উৎস অনুযায়ী accuracy — দুই dataset আলাদা করে, কোনটায় দুর্বল বোঝা যায়
src = test_df["source"].values
SOURCE_ACC = {}
for s in SOURCE_ORDER:
    m = src == s
    if m.any():
        SOURCE_ACC[s] = round(float((y_pred[m] == y_true[m]).mean()), 4)
        print(f"  {s:13s} {SOURCE_ACC[s]*100:6.2f}%  ({m.sum()}টা ছবি)")
leaf = y_true != CLASS_TO_IDX["not_leaf"]
LEAF_ACC = round(float((y_pred[leaf] == y_true[leaf]).mean()), 4)
print(f"\nশুধু ভুট্টা পাতার ছবিতে (not_leaf বাদ): {LEAF_ACC*100:.2f}%  ({leaf.sum()}টা ছবি) ← আসল মাপ")
report_txt += chr(10) + "Source accuracy: " + json.dumps(SOURCE_ACC) + chr(10) + f"Leaf-only accuracy: {LEAF_ACC}"
(OUT / "classification_report.txt").write_text(
    f"Test accuracy: {TEST_ACC*100:.2f}%" + chr(10) * 2 + report_txt, encoding="utf-8")
json.dump(classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True),
          open(OUT / "classification_report.json", "w"), indent=2)
pd.DataFrame({"path": test_df["path"].values,
              "actual": [CLASSES[i] for i in y_true],
              "predicted": [CLASSES[i] for i in y_pred],
              "confidence": conf}).to_csv(OUT / "test_predictions.csv", index=False)

cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUT / "confusion_matrix.csv")
fig, ax = plt.subplots(figsize=(10, 9))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASSES))); ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right"); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); plt.tight_layout()
savefig("04_confusion_matrix.png"); plt.show()

In [ ]:
# Confidence threshold ঠিক করা: threshold-এর ওপরে কতগুলো ছবিতে মত দেয় আর সেগুলো কত % ঠিক
thr_rows = []
print(f"{'threshold':>10} {'মত দেয় %':>10} {'সঠিক %':>10}")
for t in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    covered = m.mean() * 100
    correct = (y_pred[m] == y_true[m]).mean() * 100 if m.any() else float("nan")
    print(f"{t:>10} {covered:>10.1f} {correct:>10.1f}")
    thr_rows.append({"threshold": t, "covered_percent": round(covered, 1), "correct_percent": round(correct, 1)})
pd.DataFrame(thr_rows).to_csv(OUT / "confidence_thresholds.csv", index=False)

# server-এর threshold val দিয়ে ঠিক করা (test দিয়ে নয়, যাতে test সৎ থাকে):
# ০.৬ থেকে শুরু করে সবচেয়ে ছোট t যেখানে t-এর ওপরের ছবিগুলোর ≥৯৮% সঠিক।
# ০.৬-এর নিচে নয়: খেতের ছবি dataset-এর ছবির চেয়ে কঠিন, val-এ নিচু t ভালো দেখালেও মাঠে ঝুঁকি
val_probs = model.predict(val_ds, verbose=0)
v_pred, v_conf = val_probs.argmax(1), val_probs.max(1)
CONFIDENT_T = 0.6
for t in np.arange(0.60, 0.91, 0.05):
    mask = v_conf >= t
    if mask.any() and (v_pred[mask] == y_val[mask]).mean() >= 0.98:
        CONFIDENT_T = round(float(t), 2)
        break
MIN_T = round(max(0.40, CONFIDENT_T - 0.2), 2)
print(f"\nServer threshold → নিশ্চিত: ≥{CONFIDENT_T} | সম্ভাব্য: ≥{MIN_T}")

In [ ]:
# মাঠের কঠিন অবস্থার পরীক্ষা: অন্ধকার/ঝাপসা/noise যোগ করে test ছবি আবার চালানো
def degrade(x, y):
    x = tf.image.random_brightness(x, 50.0)
    x = tf.image.random_contrast(x, 0.6, 1.4)
    x = x + tf.random.normal(tf.shape(x), stddev=12.0)
    return tf.clip_by_value(x, 0., 255.), y

hard_ds = make_ds(test_df, False).map(degrade)
hard_probs = model.predict(hard_ds, verbose=0)
hard_acc = (hard_probs.argmax(1) == y_true).mean() * 100
print(f"পরিষ্কার test: {(y_pred == y_true).mean()*100:.1f}%  |  খারাপ অবস্থার test: {hard_acc:.1f}%")
HARD_ACC = float(hard_acc) / 100
print(f"খারাপ অবস্থায় শুধু ভুট্টা পাতা: {(hard_probs.argmax(1)[leaf] == y_true[leaf]).mean()*100:.1f}%")

## ১০. কোথায় ভুল করছে দেখুন

In [ ]:
wrong = np.where(y_pred != y_true)[0]
pd.DataFrame({"path": test_df.iloc[wrong]["path"].values,
              "actual": [CLASSES[i] for i in y_true[wrong]],
              "predicted": [CLASSES[i] for i in y_pred[wrong]],
              "confidence": conf[wrong]}).to_csv(OUT / "wrong_predictions.csv", index=False)
print("ভুল:", len(wrong), "/", len(y_true))
show = wrong[:12]
if len(show):
    cols = 4
    rows = int(np.ceil(len(show) / cols))
    plt.figure(figsize=(12, 3.2 * rows))
    for k, i in enumerate(show):
        plt.subplot(rows, cols, k + 1)
        plt.imshow(Image.open(test_df.iloc[i]["path"]).convert("RGB"))
        plt.title(f"সঠিক: {CLASSES[y_true[i]]}\nবলেছে: {CLASSES[y_pred[i]]} ({conf[i]:.0%})", fontsize=8)
        plt.axis("off")
    plt.tight_layout(); savefig("05_wrong_predictions.png"); plt.show()

## ১১. TFLite-এ রূপান্তর (float16, ছোট size)

In [ ]:
SAVED = str(WORK / "saved_model")
if os.path.exists(SAVED):
    shutil.rmtree(SAVED)
try:
    model.export(SAVED)                       # Keras 3
except AttributeError:
    model.save(SAVED, save_format="tf")       # Keras 2

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_bytes = converter.convert()

SERVER_DIR = WORK / "maize"            # server-এ ml_backend/models/maize/ হবে
SERVER_DIR.mkdir(exist_ok=True)
TFLITE_PATH = SERVER_DIR / "model.tflite"
TFLITE_PATH.write_bytes(tflite_bytes)
print(f"maize/model.tflite: {len(tflite_bytes)/1e6:.2f} MB")

In [ ]:
# TFLite model Keras-এর মতোই ঠিক চলছে কি না যাচাই (server-এ যা চলবে সেটাই)
interp = tf.lite.Interpreter(model_path=str(TFLITE_PATH))
interp.allocate_tensors()
inp_d = interp.get_input_details()[0]
out_d = interp.get_output_details()[0]
print("Input :", inp_d["shape"], inp_d["dtype"])
print("Output:", out_d["shape"], out_d["dtype"])

correct = 0
for p, y in zip(test_df["path"], test_df["y"]):
    img, _ = load_image(p, 0)
    interp.set_tensor(inp_d["index"], img.numpy()[None, ...].astype(np.float32))
    interp.invoke()
    pred = int(np.argmax(interp.get_tensor(out_d["index"])[0]))
    correct += int(pred == y)
TFLITE_ACC = correct / len(test_df)
print(f"TFLite test accuracy: {correct/len(test_df)*100:.2f}%  (Keras, TTA ছাড়া: {(probs_plain.argmax(1)==y_true).mean()*100:.2f}%)")

## ১২. Server-এর জন্য file তৈরি

In [ ]:
# app-এর maize_disease_info.dart-এর নামের সাথে মিল রাখা
BN = {
    "gray_leaf_spot": "ধূসর পাতার দাগ",
    "common_rust": "মরিচা রোগ",
    "northern_leaf_blight": "পাতা ঝলসানো রোগ",
    "fall_armyworm": "ফল আর্মিওয়ার্ম (লেদা পোকা)",
    "bacterial_leaf_streak": "ব্যাকটেরিয়াজনিত পাতার রেখা রোগ",
    "maize_chlorotic_mottle_virus": "ক্লোরোটিক মটল ভাইরাস",
    "healthy": "সুস্থ ভুট্টা গাছ",
    "not_leaf": "ভুট্টার পাতা নয়",
}

(SERVER_DIR / "labels.txt").write_text("\n".join(CLASSES), encoding="utf-8")

meta = {
    "model": BEST_NAME,
    "model_version": f"maize-{BEST_NAME}",
    "input_size": IMG,
    "input_dtype": "float32",
    "input_range": "0-255 (scale ভেতরে করা আছে)",
    "color_order": "RGB",
    "classes": CLASSES,
    "classes_bn": [BN.get(c, c) for c in CLASSES],
    "confident_threshold": CONFIDENT_T,
    "min_threshold": MIN_T,
    "test_accuracy": round(float((y_pred == y_true).mean()), 4),
    "test_accuracy_by_source": SOURCE_ACC,
    "leaf_test_accuracy": LEAF_ACC,
}
(SERVER_DIR / "meta.json").write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")

# খুললেই maize/ folder — সরাসরি ml_backend/models/-এ রাখুন
with zipfile.ZipFile(WORK / "maize_server_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["model.tflite", "labels.txt", "meta.json"]:
        z.write(SERVER_DIR / f, f"maize/{f}")

# ---------- সব ফলাফল এক zip-এ ----------
metrics = {
    "best_model": BEST_NAME,
    "tta": bool(USE_TTA_FINAL),
    "keras_test_accuracy": round(TEST_ACC, 4),
    "test_accuracy_by_source": SOURCE_ACC,
    "leaf_test_accuracy": LEAF_ACC,
    "comparison": comparison.to_dict(orient="records"),
    "tflite_test_accuracy": round(float(TFLITE_ACC), 4),
    "hard_conditions_test_accuracy": round(HARD_ACC, 4),
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "classes": CLASSES,
}
(OUT / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
(OUT / "config.json").write_text(json.dumps({
    "IMG": IMG, "BATCH": BATCH, "EPOCHS_HEAD": EPOCHS_HEAD, "EPOCHS_FT": EPOCHS_FT,
    "CANDIDATES": CANDIDATES, "LABEL_SMOOTHING": LABEL_SMOOTHING, "MIXUP_ALPHA": MIXUP_ALPHA,
    "SPECS": {k: {"unfreeze": v["unfreeze"], "lr_ft": v["lr_ft"]} for k, v in SPECS.items()}, "DROP_CLASSES": DROP_CLASSES, "SEED": SEED,
    "DATASETS": DATASETS, "NOT_LEAF_DIR": NOT_LEAF_DIR, "MAX_SIDE": MAX_SIDE,
    "NOT_LEAF_MAX": NOT_LEAF_MAX, "OTHER_CROPS_AS_NOT_LEAF": OTHER_CROPS_AS_NOT_LEAF, "OTHER_CROP_PER_FOLDER": OTHER_CROP_PER_FOLDER,
    "PV_BASE": PV_BASE, "NEAR_DUP_BITS": NEAR_DUP_BITS,
    "n_duplicates_dropped": N_DUP, "n_unreadable": N_BAD,
    "ORB_MIN_INLIERS": ORB_MIN_INLIERS, "MAX_PER_GROUP": MAX_PER_GROUP, "n_same_leaf_pairs": N_SAME,
    "CONFIDENT_T": CONFIDENT_T, "MIN_T": MIN_T,
    "RESUME": RESUME, "RESUME_MODEL": RESUME_MODEL,
}, ensure_ascii=False, indent=2), encoding="utf-8")
shutil.copytree(SERVER_DIR, OUT / "maize", dirs_exist_ok=True)
if os.path.exists(BEST):
    shutil.copy(BEST, OUT / "maize_best.keras")

ALL_ZIP = WORK / "maize_all_outputs.zip"
with zipfile.ZipFile(ALL_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
print("maize_all_outputs.zip এর ভেতরে:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT), round(p.stat().st_size / 1e6, 3), "MB")
print()

print(open(SERVER_DIR / "labels.txt", encoding="utf-8").read())
print()
for f in ["maize/model.tflite", "maize/labels.txt", "maize/meta.json",
          "maize_server_bundle.zip", "maize_all_outputs.zip"]:
    print(f, round((WORK / f).stat().st_size / 1e6, 3), "MB")

## ✅ শেষ ধাপ
Kaggle-এর **Output** ট্যাব থেকে দুটো zip download করুন:
- **`maize_server_bundle.zip`** — খুলে ভেতরের `maize/` folder-টা project-এর **`ml_backend/models/`**-এ রাখুন। এতে আছে `model.tflite` (~৬ MB), `labels.txt`, `meta.json` (threshold সহ)।
- **`maize_all_outputs.zip`** — সব chart, report, confusion matrix, ভুল prediction, আর `maize_best.keras`।

তারপর server চালিয়ে পরীক্ষা করুন:
```
python -m ml_backend.api.main
curl -F "crop=maize" -F "file=@leaf.jpg" http://localhost:8000/predict
```
`/health`-এ `"maize": true` দেখালে app-এ ভুট্টা বাছলে "সার্ভার চালু" দেখাবে।

**মনে রাখবেন**
- মোট accuracy-তে not_leaf (সহজ, অনেক ছবি) মিশে থাকে — **"শুধু ভুট্টা পাতার ছবিতে"** সংখ্যাটা দেখুন। প্রতিটা dataset-এর accuracy-ও আলাদা আছে।
- Gray Leaf Spot ↔ Northern Leaf Blight দুটোই লম্বাটে দাগ; confusion matrix-এ এদের দিকে নজর দিন।
- নিজের খেতের ভুট্টা পাতার ছবি দিয়েও একবার পরীক্ষা করুন।